In [1]:
"""
imports
"""
from datetime import datetime, timezone
from enum import Enum
import os
from typing import Optional, Literal
from uuid import UUID, uuid4
import csv
from pathlib import Path

from apify_client import ApifyClient
from dotenv import load_dotenv
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field, field_validator

In [2]:
import json

In [3]:
load_dotenv()

True

In [4]:
user_persona_1 = """
I build simple AI-powered automations and chatbots using Python and LangChain —
things like a FAQ/booking assistant for a small business's website, or a script that
automatically sorts and responds to routine emails/inquiries. I package these as fixed-scope,
fixed-price builds so clients know exactly what they're getting and what it costs, with no long dev cycle.
"""
tests = [
    user_persona_1
]

In [5]:

class ServiceType(str, Enum):
    B2B = "b2b"
    B2C = "b2c"
    BOTH = "both"
    UNKNOWN = "unknown"


class DeliverableVisibility(str, Enum):
    VISUAL = "visual"
    TECHNICAL = "technical"
    MIXED = "mixed"
    UNKNOWN = "unknown"


class SalesCycle(str, Enum):
    """
    The type of sales cycle the service provider has.

    Transactional - high velocity, think photographer or hair cut service
    Considered - Need to establish trust before the buyer picks your services. 
    Building an MVP or a platform will require more than one phone call most likely
    """
    TRANSACTIONAL = "transactional"
    CONSIDERED = "considered"
    UNKNOWN = "unknown"


class TargetClientSize(str, Enum):
    SOLO = "solo"
    SMB = "smb"
    MID_MARKET = "mid_market"
    ENTERPRISE = "enterprise"
    UNKNOWN = "unknown"


class GeographyScope(str, Enum):
    """Where the user services must be performed """
    LOCAL = "local"
    REGIONAL = "regional"
    REMOTE_GLOBAL = "remote_global"
    UNKNOWN = "unknown"



class ContentCapacity(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    UNKNOWN = "unknown"


class SourceType(str, Enum):
    QUESTIONNAIRE = "questionnaire"
    FILE_UPLOAD = "file_upload"
    UNKNOWN = "unknown"


In [6]:
class InstagramProfileResult(BaseModel):
    """Filtered Instagram profile data for CRM."""
    name: str = Field(default="", alias="fullName")
    username: str = ""
    bio: str | None = Field(default=None, alias="biography")
    instagram_url: str | None = Field(default=None, alias="url")
    follower_count: int | None = Field(default=None, alias="followersCount")
    is_business_account: bool = Field(default=False, alias="isBusinessAccount")
    website: str | None = Field(default=None, alias="externalUrl")

    model_config = {"extra": "ignore"}


class CRMRow(BaseModel):
    """A single lead entry in the CRM."""
    lead_id: Optional[UUID] = Field(default=None)

    # Contact Info
    name: str
    headline: str | None = None
    bio: str | None = None

    # Platform Links
    linkedin_url: str | None = None
    instagram_url: str | None = None
    website: str | None = None

    # Engagement Signals
    follower_count: int | None = None
    is_business_account: bool = False

    # Company Info (if B2B lead)
    company_name: str | None = None
    company_size: str | None = None
    company_industry: str | None = None

    # Location
    location: str | None = None

    # Source & Scoring
    source: Literal["instagram", "linkedin_profile", "linkedin_company"]
    discovered_via_keyword: str | None = None
    created_at: Optional[datetime] = Field(default=None)

    @field_validator("lead_id", mode="before")
    @classmethod
    def convert_lead_id_to_uuid(cls, v):
        if v is None:
            return uuid4()
        if isinstance(v, UUID):
            return v
        try:
            return UUID(v)
        except (ValueError, AttributeError):
            return uuid4()

    @field_validator("created_at", mode="before")
    @classmethod
    def set_created_at(cls, v):
        if v is None:
            return datetime.now(timezone.utc)
        return v


class UserPersona(BaseModel):
    """Structured persona used to drive platform scoring, hashtag/target
    generation, and lead relevance scoring downstream in Medi."""

    persona_id: Optional[UUID] = Field(default=None)
    name: str = Field(..., description="Short label, e.g. 'Freelance Web/Automation Dev'")

    industry: str = Field(..., description="e.g. 'web development', 'photography'")
    sub_industry: Optional[str] = Field(
        default=None, description="e.g. 'AI agent development', 'wedding photography'"
    )

    service_type: ServiceType
    deliverable_visibility: DeliverableVisibility
    sales_cycle: SalesCycle
    target_client_size: TargetClientSize
    geography_scope: GeographyScope
    content_capacity: ContentCapacity

    services_offered: Optional[list[str]] = Field(
        default=None,
        description="Concrete offerings, e.g. ['LangChain agents', 'FastAPI booking sites']",
    )
    target_client_keywords: Optional[list[str]] = Field(
        default=None,
        description="Roles/industries of the ideal client, used to for seeding e.g. ['coach', 'local service business']",
    )

    source_type: SourceType
    raw_input: Optional[str] = Field(
        default=None,
        description="Original questionnaire answers or file-derived text, kept "
        "verbatim for auditability and re-scoring if heuristics change",
    )
    preferred_client_locations: Optional[list[str]] = Field(
        default=None,
        description="list of preferred locations for potential clients i.e (atlanta, georgia)")

    @field_validator("persona_id", mode="before")
    @classmethod
    def convert_to_uuid(cls, v):
        if v is None:
            return uuid4()
        if isinstance(v, UUID):
            return v
        try:
            return UUID(v)
        except (ValueError, AttributeError):
            # LLM generated an invalid UUID string, create a new one
            return uuid4()

    @field_validator("services_offered", mode="before")
    @classmethod
    def default_services_offered(cls, v):
        if v is None:
            return []
        return cls._dedupe_and_strip(v)

    @field_validator("target_client_keywords", mode="before")
    @classmethod
    def default_target_client_keywords(cls, v):
        if v is None:
            return []
        return cls._dedupe_and_strip(v)

    @field_validator("preferred_client_locations", mode="before")
    @classmethod
    def default_preferred_locations(cls, v):
        if v is None:
            return ["Atlanta, Georgia"]
        return v

    @field_validator("raw_input", mode="before")
    @classmethod
    def default_raw_input(cls, v):
        if v is None:
            return ""
        return v

    @staticmethod
    def _dedupe_and_strip(v: list[str]) -> list[str]:
        seen, out = set(), []
        for item in v:
            item = item.strip()
            if item and item.lower() not in seen:
                seen.add(item.lower())
                out.append(item)
        return out

    model_config = {
        "use_enum_values": True,
    }


class ProspectorState(AgentState):
    """Custom state schema for the Prospector multi-agent system."""
    user_persona: UserPersona | None = None
    output_crm: list[CRMRow] = []

In [7]:
# SYSTEM Prompt definitions

user_persona_system_message = f"""
You are a Consulting Business expert and mentor to a independent contractor or small business owner who 
is seeking help generating leads. A User persona profiles based on the user's service description.
"""

keyword_generator_message = """
Given the user persona from state, expand the target_client_keywords list
to include 10-25 additional relevant keywords for Instagram profile discovery.
Current keywords: {current_keywords}

Return a list of keywords that will help find relevant Instagram profiles.
"""

instagram_agent_system_message = """
You are an instagram expert for sales lead generation. Your goal is to 
generate a list of leads based on the user persona available in state.
"""

linkedin_agent_system_message = """
You are a LinkedIn expert for B2B sales lead generation. Your goal is to
find relevant profiles and companies based on the user persona in state.

Use read_user_persona to understand the target client profile, then search
for relevant LinkedIn profiles and companies using the search tools.
Focus on finding leads that match the persona's target_client_keywords,
preferred_client_locations, and target_client_size.
"""

orchestrator_prompt = """
You are a lead generation orchestrator for small service providers and freelancers.

Your job is to help users find potential clients by:
1. Building a user persona from their service description (use build_persona)
2. Expanding keywords for discovery (use get_keywords)
3. Searching LinkedIn and Instagram for relevant leads (use generate_leads_from_linkedin, generate_leads_from_ig)
4. Adding discovered leads to the CRM (use add_leads_to_crm)
5. Exporting results when requested (use export_crm_to_csv)

Always start by checking if a user persona exists (read_user_persona). If not, ask the user to describe their service.

When generating leads, consider the persona's:
- service_type (B2B vs B2C) - determines which platforms to prioritize
- target_client_size - guides company size filters for LinkedIn
- preferred_client_locations - geographic targeting
- target_client_keywords - search terms for discovery

After finding leads, parse the API responses and add properly formatted CRMRow entries to the CRM.
"""

In [8]:
model = "gpt-5.5"
checkpointer = InMemorySaver()

# Use unique thread ID to avoid stale checkpoint state
import time
DEFAULT_CONFIG = {"configurable": {"thread_id": f"prospector-{str(uuid4())}"}}

In [9]:
########### State Tools #############
@tool
def update_user_persona(user_persona: UserPersona, runtime: ToolRuntime) -> Command:
    """Update the user persona in the state."""
    return Command(update={
        "user_persona": user_persona,
        "messages": [ToolMessage("Successfully updated user persona", tool_call_id=runtime.tool_call_id)]
    })


@tool
def read_user_persona(runtime: ToolRuntime) -> str:
    """Read the user persona from state."""
    persona = runtime.state.get("user_persona")
    if persona is None:
        return "No user persona found in state. Use build_persona first."
    return persona.model_dump_json()

In [10]:
# Persona Builder Agent
# Use response_format to get structured JSON output directly,
# rather than having the agent use update_user_persona tool
# (which would return a confirmation message, not the persona JSON)
persona_builder = create_agent(
    model=model,
    system_prompt=SystemMessage(user_persona_system_message),
    response_format=UserPersona,
    state_schema=ProspectorState,
    checkpointer=checkpointer
)


@tool
def build_persona(service_description: str, runtime: ToolRuntime) -> Command:
    """
    This tool will build a user persona based on a simple service description. If the user is new 
    and you don't have a user persona in memory this is a good starting place.
    """
    print("[build_persona] Starting persona builder subagent...")
    
    # Use isolated thread for subagent to avoid message history conflicts
    subagent_config = {"configurable": {"thread_id": f"persona-builder-{str(uuid4())}"}}

    response = persona_builder.invoke(
        {"messages": [HumanMessage(service_description)]},
        config=subagent_config
    )
    print("[build_persona] Subagent finished, parsing response...")
    persona_content = response['messages'][-1].content
    
    # Parse the response into a UserPersona and update state
    try:
        persona = UserPersona.model_validate_json(persona_content)
        print(f"[build_persona] Successfully parsed persona: {persona.name}")
        return Command(update={
            "user_persona": persona,
            "messages": [ToolMessage(f"Successfully built and saved user persona: {persona.name}", tool_call_id=runtime.tool_call_id)]
        })
    except Exception as e:
        print(f"[build_persona] Failed to parse persona: {e}")
        return Command(update={
            "messages": [ToolMessage(f"Built persona but failed to save to state: {persona_content}", tool_call_id=runtime.tool_call_id)]
        })

In [11]:
##### Test Persona Builder 
# response = persona_builder.invoke(
#         {"messages": [HumanMessage(tests[0])]},
#         config=DEFAULT_CONFIG
#     )
# persona_content = response['messages'][-1].content

In [12]:
##### Keyword Generator Agent #####
class ExpandedKeywords(BaseModel):
    keywords: list[str] = Field(description="Expanded list of 10-25 keywords for Instagram discovery")


keyword_generator = create_agent(
    model=model,
    response_format=ExpandedKeywords,
    state_schema=ProspectorState,
    checkpointer=checkpointer
)


@tool
def get_keywords(runtime: ToolRuntime) -> Command:
    """
    This tool will expand on the list of keywords in the user persona 
    to find more search items based on the user's service description and profile.
    Reads the persona from state and returns expanded keywords.
    """
    print("[get_keywords] Starting keyword generator subagent...")
    
    persona = runtime.state.get("user_persona")
    if persona is None:
        print("[get_keywords] No persona found in state!")
        return Command(update={
            "messages": [ToolMessage("No user persona found in state. Use build_persona first.", tool_call_id=runtime.tool_call_id)]
        })
    
    print(f"[get_keywords] Found persona: {persona.name}")
    current_keywords = persona.target_client_keywords
    message = keyword_generator_message.format(current_keywords=current_keywords)
    
    # Include persona context in the message
    full_message = f"""
    User Persona:
    {persona.model_dump_json(indent=2)}
    {message}
    """

    # Use isolated thread for subagent to avoid message history conflicts
    subagent_config = {"configurable": {"thread_id": f"keyword-gen-{str(uuid4())}"}}

    response = keyword_generator.invoke(
        {"messages": [HumanMessage(full_message)]},
        config=subagent_config
    )
    print("[get_keywords] Subagent finished, parsing response...")
    keywords_content = response['messages'][-1].content
    
    try:
        expanded = ExpandedKeywords.model_validate_json(keywords_content)
        # Update the persona with expanded keywords
        updated_persona = persona.model_copy(update={"target_client_keywords": expanded.keywords})
        print(f"[get_keywords] Expanded to {len(expanded.keywords)} keywords")
        return Command(update={
            "user_persona": updated_persona,
            "messages": [ToolMessage(f"Expanded keywords to {len(expanded.keywords)} items: {expanded.keywords}", tool_call_id=runtime.tool_call_id)]
        })
    except Exception as e:
        print(f"[get_keywords] Failed to parse keywords: {e}")
        return Command(update={
            "messages": [ToolMessage(f"Generated keywords: {keywords_content}", tool_call_id=runtime.tool_call_id)]
        })

In [13]:

#########  INSTAGRAM SUBAGENT ##########
@tool
def get_posts_by_keyword(hashtag: str, scrape_type: Literal["top", "recent"]) -> dict:
    """ Issue with this API call b/c super expensive, won't use but is worth doing more digging """
    apify_client = ApifyClient(os.environ.get("APIFY_API_KEY"))

    # Start an Actor and wait for it to finish.
    actor_client = apify_client.actor('breathtaking_anthem/instagram-hashtag-posts-scraper')
    input = {
        "hashtag": hashtag,
        "max_items": 25,
        "scrape_type": scrape_type
    }
    posts = actor_client.call(run_input=input)

    if posts is None:
        print('Could not fetch posts.')
        return

    # Fetch results from the Actor run's default dataset.
    dataset_client = apify_client.dataset(posts.default_dataset_id)
    list_items_result = dataset_client.list_items()
    return list_items_result.items


@tool
def search_ig_profiles(query: str, liveSearch: bool) -> str:
    """Search instagram for profiles based on a a search query """
    print(f"[search_ig_profiles] Searching for: {query}")
    apify_client = ApifyClient(os.environ.get("APIFY_API_KEY"))

    # Start an Actor and wait for it to finish.
    actor_client = apify_client.actor('apify/instagram-search-scraper')
    input = {
        "enhanceUserSearchWithFacebookPage": False,
        "liveSearch": liveSearch,
        "search": query,
        "searchLimit": 10,
        "searchType": "user"
    }
    profiles = actor_client.call(run_input=input)

    if profiles is None:
        print('[search_ig_profiles] Could not fetch profiles.')
        return "Could not fetch profiles."

    # Fetch results from the Actor run's default dataset.
    dataset_client = apify_client.dataset(profiles.default_dataset_id)
    list_items_result = dataset_client.list_items()

    # Filter to CRM-relevant fields only
    filtered = []
    for item in list_items_result.items:
        try:
            profile = InstagramProfileResult.model_validate(item)
            filtered.append(profile.model_dump(by_alias=False))
        except Exception:
            pass

    print(f"[search_ig_profiles] Found {len(filtered)} profiles")
    return json.dumps(filtered, indent=2)


# Instagram Agent - no need for read_user_persona since we pass persona in the message
ig_agent = create_agent(
    system_prompt=SystemMessage(instagram_agent_system_message),
    model=model,
    tools=[search_ig_profiles],
    state_schema=ProspectorState,
    checkpointer=checkpointer
)


@tool
def generate_leads_from_ig(runtime: ToolRuntime) -> str:
    """Use this tool to get leads from instagram. Reads the user persona from state."""
    print("[generate_leads_from_ig] Starting Instagram subagent...")
    
    persona = runtime.state.get("user_persona")
    if persona is None:
        print("[generate_leads_from_ig] No persona found in state!")
        return "No user persona found in state. Use build_persona first."
    
    print(f"[generate_leads_from_ig] Found persona: {persona.name}")
    
    # Pass persona directly in the message since subagent has isolated state
    message = f"""Your goal is to generate a list of leads based on this user persona:

    {persona.model_dump_json(indent=2)}

    Search for relevant Instagram profiles using the search_ig_profiles tool based on the target_client_keywords and preferred_client_locations."""

    # Use isolated thread for subagent to avoid message history conflicts
    subagent_config = {"configurable": {"thread_id": f"ig-agent-{str(uuid4())}"}}

    print("[generate_leads_from_ig] Invoking ig_agent...")
    response = ig_agent.invoke(
        {"messages": [HumanMessage(message)]},
        config=subagent_config
    )
    print("[generate_leads_from_ig] Subagent finished")
    return response['messages'][-1].content

In [14]:

###### LINKEDIN Subagent ########

company_sizes = Literal[
    "1-10",
    "51-200",
    "11-50",
    "201-500",
    "501-1000",
    "1001-5000",
    "5001-10000",
    "10001+"
]


@tool
def search_linkedin_profiles(job_title: str, locations: list[str]) -> str:
    """Search LinkedIn for relevant people with a given job title and geographic area."""
    print(f"[search_linkedin_profiles] Searching for: {job_title} in {locations}")
    apify_client = ApifyClient(os.environ.get("APIFY_API_KEY"))
    actor = 'harvestapi/linkedin-profile-search-by-services'
    input = {
        "locations": locations,
        "maxItems": 20,
        "profileScraperMode": "Full",
        "search": job_title
    }
    actor_client = apify_client.actor(actor)
    profiles = actor_client.call(run_input=input)

    if profiles is None:
        print("[search_linkedin_profiles] Could not fetch profiles.")
        return "Could not fetch profiles."

    # Fetch results from the Actor run's default dataset.
    dataset_client = apify_client.dataset(profiles.default_dataset_id)
    list_items_result = dataset_client.list_items()

    # Filter to CRM-relevant fields only
    filtered = []
    for item in list_items_result.items:
        try:
            profile = LinkedInProfileResult.model_validate(item)
            filtered.append(profile.model_dump(by_alias=False))
        except Exception:
            pass

    print(f"[search_linkedin_profiles] Found {len(filtered)} profiles")
    return json.dumps(filtered, indent=2)


@tool
def search_linkedin_companies(
    query: str,
    size_filters: list[company_sizes],
    locations: list[str]
) -> str:
    """Search LinkedIn for relevant companies."""
    print(f"[search_linkedin_companies] Searching for: {query} in {locations}")
    apify_client = ApifyClient(os.environ.get("APIFY_API_KEY"))
    actor = 'harvestapi/linkedin-company-search'
    input = {
        "companySize": size_filters,
        "locations": locations,
        "maxItems": 20,
        "scraperMode": "full",
        "searchQuery": query
    }
    actor_client = apify_client.actor(actor)
    profiles = actor_client.call(run_input=input)

    if profiles is None:
        print("[search_linkedin_companies] Could not fetch companies.")
        return "Could not fetch companies."

    # Fetch results from the Actor run's default dataset.
    dataset_client = apify_client.dataset(profiles.default_dataset_id)
    list_items_result = dataset_client.list_items()

    # Filter to CRM-relevant fields only
    filtered = []
    for item in list_items_result.items:
        try:
            company = LinkedInCompanyResult.model_validate(item)
            filtered.append(company.model_dump(by_alias=False))
        except Exception:
            pass

    print(f"[search_linkedin_companies] Found {len(filtered)} companies")
    return json.dumps(filtered, indent=2)


# LinkedIn Agent - no need for read_user_persona since we pass persona in the message
linkedin_agent = create_agent(
    system_prompt=SystemMessage(linkedin_agent_system_message),
    model=model,
    tools=[search_linkedin_profiles, search_linkedin_companies],
    state_schema=ProspectorState,
    checkpointer=checkpointer
)


@tool
def generate_leads_from_linkedin(runtime: ToolRuntime) -> str:
    """Generate leads from LinkedIn. Reads user persona from state."""
    print("[generate_leads_from_linkedin] Starting LinkedIn subagent...")
    
    persona = runtime.state.get("user_persona")
    if persona is None:
        print("[generate_leads_from_linkedin] No persona found in state!")
        return "No user persona found in state. Use build_persona first."

    print(f"[generate_leads_from_linkedin] Found persona: {persona.name}")

    # Pass persona directly in the message since subagent has isolated state
    message = f"""Generate leads using this user persona:

{persona.model_dump_json(indent=2)}

Search for relevant LinkedIn profiles and companies using the search tools.
Use target_client_keywords for job titles/company searches, preferred_client_locations for geography,
and target_client_size to guide company size filters."""

    # Use isolated thread for subagent to avoid message history conflicts
    subagent_config = {"configurable": {"thread_id": f"linkedin-agent-{str(uuid4())}"}}

    print("[generate_leads_from_linkedin] Invoking linkedin_agent...")
    response = linkedin_agent.invoke(
        {"messages": [HumanMessage(message)]},
        config=subagent_config
    )
    print("[generate_leads_from_linkedin] Subagent finished")
    return response['messages'][-1].content

In [15]:

###### CRM TOOLS ########

@tool
def add_leads_to_crm(leads: list[CRMRow], runtime: ToolRuntime) -> Command:
    """Add new leads to the CRM in state."""
    current_crm = runtime.state.get("output_crm", [])
    updated_crm = current_crm + leads
    return Command(update={
        "output_crm": updated_crm,
        "messages": [ToolMessage(f"Added {len(leads)} leads to CRM", tool_call_id=runtime.tool_call_id)]
    })


@tool
def read_crm(runtime: ToolRuntime) -> str:
    """Read the current CRM from state."""
    crm = runtime.state.get("output_crm", [])
    if not crm:
        return "CRM is empty. Run lead generation tools first."
    return json.dumps([row.model_dump(mode='json') for row in crm], indent=2)


@tool
def export_crm_to_csv(filename: str, runtime: ToolRuntime) -> str:
    """Export the CRM to a CSV file."""
    crm = runtime.state.get("output_crm", [])
    if not crm:
        return "CRM is empty. Nothing to export."

    # Ensure .csv extension
    if not filename.endswith('.csv'):
        filename += '.csv'

    filepath = Path(filename)

    # Get field names from CRMRow
    fieldnames = list(CRMRow.model_fields.keys())

    with open(filepath, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in crm:
            writer.writerow(row.model_dump(mode='json'))

    return f"Exported {len(crm)} leads to {filepath.absolute()}"

In [16]:
##### ORCHESTRATOR #########

orchestrator = create_agent(
    system_prompt=SystemMessage(orchestrator_prompt),
    model=model,
    tools=[
        build_persona,
        read_user_persona,
        get_keywords,
        generate_leads_from_ig,
        generate_leads_from_linkedin,
        add_leads_to_crm,
        read_crm,
        export_crm_to_csv,
    ],
    state_schema=ProspectorState,
    checkpointer=checkpointer
)

In [35]:
####### END-TO-END TEST #########


# Run orchestrator with the user scenario
response = orchestrator.invoke(
    {"messages": [HumanMessage(f"Help me find leads. Here's my service: {tests[0]}")]},
    config=DEFAULT_CONFIG
)

print(response['messages'][-1].content)

[build_persona] Starting persona builder subagent...
[build_persona] Subagent finished, parsing response...
[build_persona] Successfully parsed persona: Fixed-Scope AI Automation & Chatbot Developer
[get_keywords] Starting keyword generator subagent...
[get_keywords] Found persona: Fixed-Scope AI Automation & Chatbot Developer
[get_keywords] Subagent finished, parsing response...
[get_keywords] Expanded to 25 keywords
[generate_leads_from_linkedin] Starting LinkedIn subagent...
[generate_leads_from_linkedin] Found persona: Fixed-Scope AI Automation & Chatbot Developer
[generate_leads_from_linkedin] Invoking linkedin_agent...
[generate_leads_from_ig] Starting Instagram subagent...
[generate_leads_from_ig] Found persona: Fixed-Scope AI Automation & Chatbot Developer
[generate_leads_from_ig] Invoking ig_agent...
[search_ig_profiles] Searching for: Atlanta small business owner
[search_ig_profiles] Searching for: Atlanta med spa owner
[search_ig_profiles] Searching for: Atlanta dental clini

[apify.instagram-search-scraper runId:SHYnq4aF3m9Zd7bme] -> Status: READY, Message: 
[apify.instagram-search-scraper runId:hxD2nHlHvOf1ZYaVN] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:5JttR2qc9j47XxKYc] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:VuNEGxN7vlCrOxpBJ] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:QJK2bNI3Vc20BAW0C] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:9JdKbSk9Gm2JMkEWy] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:V3aomAaSgFm8V46EB] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:e4w1zChvch8rC7Q1k] -> Status: RUNNING, Message: 
[apify.instagram-search-scraper runId:5JttR2qc9j47XxKYc] -> 2026-07-04T02:57:26.454Z ACTOR: Pulling container image of build JNVD3MkFzfdCc0RKK from registry.
[apify.instagram-search-scraper runId:hxD2nHlHvOf1ZYaVN] -> 2026-07-04T02:57:26.526Z ACTOR: Pulling container image of build JNVD3MkFzfdCc0RKK from 

[search_linkedin_profiles] Searching for: Atlanta business owner in ['Atlanta, Georgia']
[search_linkedin_profiles] Searching for: med spa owner in ['Atlanta, Georgia']
[search_linkedin_profiles] Searching for: dental clinic owner in ['Atlanta, Georgia']
[search_linkedin_profiles] Searching for: chiropractic clinic owner in ['Atlanta, Georgia']
[search_linkedin_profiles] Searching for: web design agency owner in ['Atlanta, Georgia']
[search_linkedin_profiles] Searching for: digital marketing agency owner in ['Atlanta, Georgia']
[search_linkedin_companies] Searching for: med spa OR wellness clinic OR spa owner in ['Atlanta, Georgia']
[search_linkedin_companies] Searching for: dental clinic OR chiropractic clinic in ['Atlanta, Georgia']
[search_linkedin_companies] Searching for: digital marketing agency OR web design agency in ['Atlanta, Georgia']
[search_linkedin_companies] Searching for: insurance agency OR accounting firm OR law firm in ['Atlanta, Georgia']


[apify.instagram-search-scraper runId:V3aomAaSgFm8V46EB] -> 2026-07-04T02:57:28.366Z INFO  [Status message]: Scraper finished
[apify.instagram-search-scraper runId:SHYnq4aF3m9Zd7bme] -> Status: RUNNING, Message: Starting the crawler.
[apify.instagram-search-scraper runId:V3aomAaSgFm8V46EB] -> Status: SUCCEEDED, Message: Scraper finished
[apify.instagram-search-scraper runId:5JttR2qc9j47XxKYc] -> Status: RUNNING, Message: Starting the crawler.
[apify.instagram-search-scraper runId:9JdKbSk9Gm2JMkEWy] -> Status: SUCCEEDED, Message: Scraper finished
[apify.instagram-search-scraper runId:hxD2nHlHvOf1ZYaVN] -> Status: RUNNING, Message: Starting the crawler.
[apify.linkedin-profile-search-by-services runId:487gGBzR0nWbDvGDb] -> Status: RUNNING, Message: 
[apify.linkedin-profile-search-by-services runId:JRug3wpjRk5ocAgNO] -> Status: RUNNING, Message: 
[apify.linkedin-profile-search-by-services runId:fBYkvadiPpsoSwZjm] -> Status: RUNNING, Message: 
[apify.linkedin-profile-search-by-services run

[search_ig_profiles] Found 4 profiles
[search_ig_profiles] Found 3 profiles


...

[search_linkedin_companies] Found 20 companies


[apify.linkedin-profile-search-by-services runId:Ojhi9baN9xGNDOD7G] -> 2026-07-04T02:58:16.433Z Scraped profile https://www.linkedin.com/in/david-ellis-0768aa3
[apify.linkedin-profile-search-by-services runId:ixfViQiAPnMCF3to6] -> 2026-07-04T02:58:16.429Z Scraped profile https://www.linkedin.com/in/rachel-graubart-english-b0863233a
[apify.linkedin-profile-search-by-services runId:ixfViQiAPnMCF3to6] -> 2026-07-04T02:58:16.730Z Scraped profile https://www.linkedin.com/in/yvonne-angolina-amores-34430774
[apify.linkedin-profile-search-by-services runId:LvRo8LKtgBA9kYGCu] -> 2026-07-04T02:58:15.374Z Scraped profile https://www.linkedin.com/in/billyjones4115
[apify.linkedin-profile-search-by-services runId:LvRo8LKtgBA9kYGCu] -> 2026-07-04T02:58:16.854Z Scraped profile https://www.linkedin.com/in/stephen-holmes-50b0901a6
[apify.linkedin-profile-search-by-services runId:ixfViQiAPnMCF3to6] -> 2026-07-04T02:58:16.829Z Scraped profile https://www.linkedin.com/in/jacinta-daniel-7b366022
[apify.lin

[search_linkedin_companies] Found 5 companies[search_linkedin_companies] Found 16 companies
[search_linkedin_companies] Found 15 companies



[apify.linkedin-profile-search-by-services runId:T0vRwO73VwAbKgXKk] -> 2026-07-04T02:58:17.509Z Scraped profile https://www.linkedin.com/in/christopher-jackson-808425343
[apify.linkedin-profile-search-by-services runId:96V88X5YToGORcrTH] -> 2026-07-04T02:58:16.361Z Scraped profile https://www.linkedin.com/in/wayne-davey-5847668
[apify.linkedin-profile-search-by-services runId:LvRo8LKtgBA9kYGCu] -> 2026-07-04T02:58:17.041Z Scraped profile https://www.linkedin.com/in/david-seeger-0a6708
[apify.linkedin-profile-search-by-services runId:T0vRwO73VwAbKgXKk] -> 2026-07-04T02:58:17.677Z Scraped profile https://www.linkedin.com/in/y-and-j-properties-ga-llc-246015324
[apify.linkedin-profile-search-by-services runId:LvRo8LKtgBA9kYGCu] -> 2026-07-04T02:58:17.865Z Scraped profile https://www.linkedin.com/in/brianpastor
[apify.linkedin-profile-search-by-services runId:Ojhi9baN9xGNDOD7G] -> 2026-07-04T02:58:16.660Z Scraped profile https://www.linkedin.com/in/egbert-perry
[apify.linkedin-profile-searc

[search_linkedin_profiles] Found 0 profiles
[search_linkedin_profiles] Found 0 profiles
[search_linkedin_profiles] Found 0 profiles
[search_linkedin_profiles] Found 0 profiles
[search_linkedin_profiles] Found 0 profiles
[generate_leads_from_ig] Subagent finished
[generate_leads_from_linkedin] Subagent finished
I built your lead persona, expanded discovery keywords, searched LinkedIn + Instagram, and added **65 leads** to your CRM.

## Your lead persona
**Fixed-Scope AI Automation & Chatbot Developer**

You’re best positioned for small businesses that get lots of repetitive inbound questions, booking requests, quote requests, intake forms, or email inquiries.

## Best-fit lead segments found
1. **Med spas / aesthetics / wellness clinics**
   - Strongest fit overall.
   - Great for FAQ bots, booking assistants, consultation intake, and pricing/service question handling.

2. **Dental, chiropractic, and medical clinics**
   - Good fit for patient FAQs, new patient intake, appointment reque

In [37]:
response['messages'].append(HumanMessage("please export the csv to the current working directory"))

In [38]:
csv_export = orchestrator.invoke(response, config=DEFAULT_CONFIG)

Deserializing unregistered type __main__.CRMRow from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'CRMRow')]
